# The data agent, step by step

This notebook redoes **by hand, step by step**, what the `fetch_prices` function in `data/data_agent.py` does.
It does not contain the agent's code: it **calls** it. The real code stays in `data/data_agent.py`.

Run the cells one at a time with **Shift + Enter**.

## The big picture

The data agent answers one question: **what was the closing price of each of the 50 ETFs, every day, in euros, on the exchange Belfius uses?**

It gets there in 4 steps, each one solving a problem:

| Step | The problem | The solution |
|---|---|---|
| 1. Mapping | Belfius says `WORLD`, Yahoo only knows `SXRT.DE` | A table that links the two |
| 2. Download | We need the prices of the 50 ETFs | One call to Yahoo Finance |
| 3. Calendar | Exchanges do not share the same holidays | Put all ETFs on the same days |
| 4. Names | The rest of the project uses Belfius names | Rename the columns |

Step 5 runs the full `fetch_prices` function and checks that it gives the same result as steps 1 to 4 done by hand.

## 0. Setup

Tell Python where the project is (the folder above `notebooks/`), so that `data.data_agent` can be imported.
`autoreload` reloads `data_agent.py` automatically when it changes.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath(".."))

from datetime import date
import pandas as pd
from data.universe import load_universe, reference_exchange, yahoo_suffix
from data.check_etfs_yfinance import candidates, describe, pick_best, build_yahoo_map, YAHOO_MAP_CSV
from data.data_agent import load_yahoo_map, yahoo_download, fetch_prices

start, end = date(2025, 1, 1), date.today()

## 1. Finding the right Yahoo symbol for each ETF

**The problem.** Belfius gave us a list of 50 ETFs, each with a short name (`WORLD`) and an ISIN (`IE00B4L5Y983`).
The ISIN is the unique international identifier of a fund. But Yahoo Finance does not work with ISINs: it needs a symbol, like `SXRT.DE`.
And one ETF has several symbols, one per exchange where it is listed: `.DE` for Xetra, `.L` for London, `.MI` for Milan…
These listings trade at different prices, sometimes in different currencies.

**What is at stake.** According to the official Belfius Q&A, every order is executed and valued **at the closing price of one specific exchange**,
and all those listings are **in euros**. If we take the wrong listing (for example London in dollars), our prices are not the competition's prices,
and every indicator, backtest and decision built on them is slightly off.

**How we build it.** Six functions, each doing one job, from the Belfius list to the table the data agent uses:

```
load_universe()                      the 50 ETFs: Belfius name + ISIN                 (data/universe.py)
reference_exchange() / yahoo_suffix() the exchange Belfius uses for each ETF          (data/universe.py)
candidates()                         every Yahoo symbol listing one ETF               (data/check_etfs_yfinance.py)
describe()                           currency and history of one listing              (data/check_etfs_yfinance.py)
pick_best()                          the listing to keep, and its status              (data/check_etfs_yfinance.py)
build_yahoo_map()                    the three steps above for all 50 ETFs            (data/check_etfs_yfinance.py)
        |  saved once to data/etf_yahoo_map.csv  (python data/check_etfs_yfinance.py)
load_yahoo_map()                     reads that file every time we need prices        (data/data_agent.py)
```

We go down the chain one function at a time, on one example ETF.

### 1.1 `load_universe()` — the 50 ETFs

**What it does.** Reads `data/etf_universe.csv`, the list copied from the Belfius session: one row per ETF, with its theme, short name (`ticker`), ISIN and full name.

**Why.** It is the single source of truth for the universe. Every other part of the project takes its list of ETFs from here, so nobody types the list twice.

In [ ]:
universe = load_universe()
print(len(universe), "ETFs")
universe.head()

### 1.2 `reference_exchange()` and `yahoo_suffix()` — where Belfius prices each ETF

**What they do.** `reference_exchange(ticker)` gives the exchange Belfius uses for that ETF: Xetra by default, with three exceptions
(BEL 20 on Euronext Brussels, COPPER LVG and PALLADIUM LVG on Euronext Milan). `yahoo_suffix(ticker)` turns that exchange into the
ending of the Yahoo symbol: `.DE` for Xetra, `.BR` for Brussels, `.MI` for Milan.

**Why.** It encodes the Belfius rule in one place. The search below can then recognise the right listing just by its ending.

In [ ]:
for t in ["WORLD", "NASDAQ 3X", "BEL 20", "COPPER LVG"]:
    print(f"{t:12} -> {reference_exchange(t):18} Yahoo suffix {yahoo_suffix(t)}")

### 1.3 `candidates()` — every Yahoo symbol for one ETF

**What it does.** Searches the ETF's ISIN on Yahoo Finance and returns every symbol found. For four ETFs that the search does not find
on Xetra, it also adds symbols we looked up by hand (`EXTRA_CANDIDATES` at the top of the script).

**Why.** Yahoo has no direct "ISIN to price" lookup: the search is the only way from an ISIN to symbols. It often returns several
listings, and sometimes misses the one we want, hence the manual extras.

We follow one example from here: **NASDAQ 3X**, which is listed in London (USD), Milan and Xetra.

In [ ]:
example = "NASDAQ 3X"
isin = universe.set_index("ticker").loc[example, "isin"]
symbols = candidates(example, isin)
print(example, isin)
symbols

### 1.4 `describe()` — what each listing looks like

**What it does.** For one symbol, downloads one year of history and returns its currency, number of trading days and last date.
It returns nothing (`None`) when Yahoo has no data for that symbol.

**Why.** To choose between listings we need facts: is it in euros? Does it have enough history to compute our indicators?
Is it still trading? Symbols without data are dropped here.

In [ ]:
listings = [d for d in (describe(s) for s in symbols) if d]
pd.DataFrame(listings)

### 1.5 `pick_best()` — the listing we keep, and its status

**What it does.** Ranks the listings and keeps the first one, using three criteria in this order:
1. **on the Belfius reference exchange** (the symbol ends with the right suffix): those are the competition's prices;
2. **in euros**: if the reference exchange is missing, a euro listing is the closest substitute;
3. **the longest history**: more data for the indicators and the backtest.

It also gives a status: `OK` (reference exchange, at least 200 trading days over a year), `PARTIAL` (reference exchange, less history),
`OTHER EXCHANGE` (we had to fall back on another listing) or `NOT FOUND`.

**Why.** The order of the criteria is the Belfius rule first, data quality second. The status tells us which ETFs need a manual check.

In [ ]:
best, status = pick_best(example, listings)
print(status)
best

### 1.6 `build_yahoo_map()` — the same for all 50 ETFs, saved once

**What it does.** Runs `candidates` → `describe` → `pick_best` for every ETF and returns one table. Running the script
`python data/check_etfs_yfinance.py` from a terminal calls it and saves the table to `data/etf_yahoo_map.csv`.

**Why it is a separate script, run once.** It makes hundreds of calls to Yahoo and takes several minutes. The answer only changes
when the ETF list or the rules change, so we compute it once and save it, instead of searching again every time we need prices.

The cell below reads the saved file to show the result. To rebuild it here instead, uncomment the last line (several minutes).

In [ ]:
saved = pd.read_csv(YAHOO_MAP_CSV)
print(saved["status"].value_counts().to_string(), "\n")
saved[saved["status"] != "OK"]          # the ETFs to check by hand
# saved = build_yahoo_map()

### 1.7 `load_yahoo_map()` — what the data agent reads

**What it does.** Reads `data/etf_yahoo_map.csv` and keeps only what the download needs: the Belfius name (`ticker`),
the Yahoo symbol and the currency. It drops ETFs without a symbol, and assumes EUR when Yahoo did not report a currency
(the reference exchanges all trade in euros). If the file does not exist, it says to run the script first.

**Why.** `fetch_prices` calls it every time we need prices. Reading a saved file takes a fraction of a second, whereas searching Yahoo takes minutes.

**Check:** all 50 ETFs are there, each with a symbol and the currency `EUR`.

In [ ]:
ymap = load_yahoo_map()
print(len(ymap), "ETFs in the table")
ymap.head(10)

In [ ]:
# Currencies of the chosen listings: ideally all EUR
ymap["currency"].value_counts()

### What to say in the report

*"Belfius identifies each ETF by its ISIN and prices it at the close of one reference exchange, in euros (Xetra for 47 of the 50 ETFs).
Yahoo Finance, our free data source, identifies listings by symbol, and one ETF has several listings. We therefore built a mapping step:
for each ISIN we collect every Yahoo listing, describe its currency and history, and keep the listing on the Belfius reference exchange,
falling back on a euro listing and then on the longest history. Each ETF receives a status, and the few that cannot be matched automatically
are checked by hand. The result is saved once and reused by the data agent every day."*

## 2. Download the raw prices from Yahoo

One call for all 50 ETFs. We take the **real closing prices**, not adjusted for dividends,
because Belfius values the portfolio at the real close and does not credit dividends.

**Result:** one row per day, one column per Yahoo symbol.

In [ ]:
raw = yahoo_download(ymap["yahoo_symbol"].tolist(), start, end)
print(raw.shape[0], "days x", raw.shape[1], "symbols")
raw.head()

## 3. Put all ETFs on the same calendar

Exchanges do not all share the same holidays. All ETFs go on the business-day calendar,
and a day without a quote takes the previous close, **for at most 3 days**.

**Check:** almost no empty cells should remain (except at the very start, for a recently launched ETF).

In [ ]:
calendar = pd.bdate_range(start, end)
aligned = raw.reindex(calendar).ffill(limit=3)

# Days still empty, for the most affected ETFs
aligned.isna().sum().sort_values(ascending=False).head(10)

## 4. Rename with the Belfius names

Everywhere else in the project, an ETF goes by its Belfius name (`WORLD`), never by its Yahoo symbol.

In [ ]:
names = dict(zip(ymap["yahoo_symbol"], ymap["ticker"]))
prices = aligned.rename(columns=names)
prices.tail()

## 5. The final result of `fetch_prices`

The full function does steps 1 to 4 in one go, and returns the table in the format the rest of the team expects:
**one row per day and per ETF**, with the columns `date`, `ticker`, `close_eur`.

**Check:** the same number of ETFs as in step 1, and the same last prices as in step 4.

In [ ]:
df = fetch_prices(start, end)
print(df["ticker"].nunique(), "ETFs,", len(df), "rows")
df.head()

In [ ]:
# The last prices from the function and from step 4 must be identical
comparison = pd.DataFrame({
    "fetch_prices": df.groupby("ticker")["close_eur"].last(),
    "step_4": prices.ffill().iloc[-1].round(4),
})
comparison["gap"] = comparison["fetch_prices"] - comparison["step_4"]
comparison.sort_values("gap", key=abs, ascending=False).head()